# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Inspect the record sets in the dataset along with their field `@id`s for further exploration.

In [ ]:
# List all record sets and their fields (referencing by `@id`)
record_sets = metadata.recordSet
print(f"Number of record sets: {len(record_sets)}\n")
for idx, rs in enumerate(record_sets):
    print(f"RecordSet {idx+1}: @id = {getattr(rs, '@id', 'N/A')}, name = {getattr(rs, 'name', 'N/A')}")
    # List fields for this record set
    if hasattr(rs, 'field'):
        fields = rs.field
        for field in fields:
            print(f"    Field: @id = {getattr(field, '@id', 'N/A')}, name = {getattr(field, 'name', 'N/A')}")
    else:
        print("    No fields found.")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record sets and field `@id`s are used and referenced as discovered above.

In [ ]:
# Gather all record set @ids
record_set_ids = [getattr(rs, '@id') for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("No records found for this record set.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For illustration, select the largest available record set and a numeric field within it (e.g. `cr:coverage` or similar if present) using `@id` variables.

In [ ]:
# Identify a suitable record set and numeric field

# As an example, pick the record set with the most records
if len(dataframes) > 0:
    record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    df = dataframes[record_set_id]
    print(f"Selected RecordSet for EDA: {record_set_id}, shape: {df.shape}")
    print("Sample columns:", df.columns.tolist())
    # Heuristically pick a numeric field (e.g. 'cr:coverage_percent', 'cr:mw', etc.)
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]
        print(f"Numeric field selected for filtering: {numeric_field}")
        threshold = df[numeric_field].mean()  # Use mean as threshold for demonstration
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Optionally group by a categorical field
        cat_fields = df.select_dtypes(include='object').columns.tolist()
        group_field = None
        if len(cat_fields) > 0:
            group_field = cat_fields[0]
            print(f"Grouping by categorical field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print("Grouped Data:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record sets with records to analyze.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'numeric_field' in locals():
    # Histogram of numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    # Boxplot grouped by group_field (if exists)
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded metadata and explored the available record sets and fields using their unique `@id` entries.
- Extracted data, performed filtering and normalization on a numeric field, and grouped by a key attribute.
- Visualized the data distribution for effective EDA using `matplotlib` and `seaborn`.
- The notebook can be extended for more advanced analyses or for additional record sets as required.